In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, rand, floor, when
import time

In [3]:
spark = (
    SparkSession.builder
    .appName("TrueSkewBenchmark")
    .config("spark.sql.shuffle.partitions", 64)
    .config("spark.sql.adaptive.enabled", "false")  # disable AQE for baseline
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [7]:
# 100M rows
N = 100_000_000

base = spark.range(N)

# 40% of data goes to ONE key
threshold = int(N * 0.4)

skewed_fact = base.withColumn(
    "user_id",
    when(col("id") < threshold, lit(1))
    .otherwise(col("id") % 10)
)

dim = spark.range(0, 10).withColumnRenamed("id", "user_id")


In [8]:
def benchmark(name, fn):
    start = time.time()
    fn()
    print(f"{name}: {time.time() - start:.2f} sec")

benchmark(
    "Skewed Join (NO mitigation)",
    lambda: skewed_fact.join(dim, "user_id").count()
)

Skewed Join (NO mitigation): 2.03 sec


In [14]:
#random salting
salted_fact = skewed_fact.withColumn(
    "salt",
    floor(rand() * SALT_BUCKETS)
).withColumn(
    "salted_key",
    col("user_id") * SALT_BUCKETS + col("salt")
)


In [15]:
# deterministic salt expansion
salt_values = spark.range(SALT_BUCKETS).withColumnRenamed("id", "salt")

salted_dim = dim.crossJoin(salt_values).withColumn(
    "salted_key",
    col("user_id") * SALT_BUCKETS + col("salt")
)

In [17]:
benchmark(
    "Skewed Join (Manual Salting)",
    lambda: salted_fact.join(salted_dim, "salted_key").count()
)


Skewed Join (Manual Salting): 0.67 sec


In [18]:
benchmark(
    "Skewed Join (Deterministic Salting)",
    lambda: salted_fact.join(salted_dim, "salted_key").count()
)

Skewed Join (Deterministic Salting): 0.49 sec
